    In this we introduce the self-consistency modules used.

In [1]:
using LinearAlgebra
using PyCall
using Statistics
using CSV
using DataFrames
using Plots
using DelimitedFiles
@pyimport numpy as np

In [2]:
# Importing personal modules
include("src/bdg_utilities.jl")
include("src/general_utils.jl")

Main.GeneralUtils

In [3]:
# Loading personal modules
using .BdGUtilities: generate_t_map, extract_geometry, build_H_twisted, generate_t_matrix
using .GeneralUtils: fermi_fn, show_all, show_matrix, check_rel_tol

using .BdGUtilities: create_bdg_ham_full, create_bdg_ham_full_twisted, create_bdg_ham_up, create_bdg_ham_dn
using .BdGUtilities: sort_evecs_spinfull, sort_evecs

f = fermi_fn

fermi_fn (generic function with 2 methods)

In [4]:
"""
    calc_delta(U, un_up, vn_up, un_dn, vn_dn, nSites, E1, E2, T)

Onsite singlet gap update for the non-spinful split solve.
Inputs are positive-energy amplitudes only (nSites states).

Returns:
- DeltaList : Vector{Float64} if purely real, else Vector{ComplexF64}
"""
function calc_delta(
    U,
    un_up::AbstractMatrix{<:Number},
    vn_up::AbstractMatrix{<:Number},
    un_dn::AbstractMatrix{<:Number},
    vn_dn::AbstractMatrix{<:Number},
    nSites::Int,
    E1::AbstractVector{<:Number},
    E2::AbstractVector{<:Number},
    T
)
    DeltaList = zeros(ComplexF64, nSites)

    @inbounds for i in 1:nSites
        Δ = 0.0 + 0.0im
        for n in 1:nSites
            f1 = f(E1[n]; T=T)
            f2 = f(E2[n]; T=T)
            
            Δ += U * (
                un_up[n, i] * conj(vn_dn[n, i]) * (1 - f1)
                - un_dn[n, i] * conj(vn_up[n, i]) * f2
            )
        end
        DeltaList[i] = Δ
    end

    return all(isreal, DeltaList) ? Float64.(DeltaList) : DeltaList
end


"""
    calc_avg_n(un, vn, nSites, E1, E2, T)

Site-resolved density update for one spin sector in the split (non-spinful) solve.
Uses positive-energy states only (nSites states).

Returns:
- avgNList :: Vector{Float64}
"""
function calc_avg_n(un, vn, nSites, E1, E2, T)
    avgNList = zeros(Float64, nSites)

    @inbounds for i in 1:nSites
        ni = 0.0
        for n in 1:nSites
            f1 = f(E1[n]; T=T)
            f2 = f(E2[n]; T=T)

            ni += abs2(un[n, i]) * f1 + abs2(vn[n, i]) * (1 - f2)
        end
        avgNList[i] = ni
    end

    return avgNList
end



function calc_delta_spinfull(
    U,
    un_up::AbstractMatrix{<:Number},
    vn_up::AbstractMatrix{<:Number},
    un_dn::AbstractMatrix{<:Number},
    vn_dn::AbstractMatrix{<:Number},
    nSites::Int,
    Epos::AbstractVector{<:Real},
    T
)
    DeltaList = zeros(ComplexF64, nSites)
    nStates = length(Epos)
    @assert size(un_up,1) == nStates

    for i in 1:nSites
        Δ = zero(ComplexF64)
        for n in 1:nStates
            fn = f(Epos[n]; T=T)
            Δ += U * (
                un_up[n, i] * conj(vn_dn[n, i]) * (1 - fn)
                - un_dn[n, i] * conj(vn_up[n, i]) * fn
            )
        end
        DeltaList[i] = Δ
    end

    return all(isreal, DeltaList) ? Float64.(DeltaList) : DeltaList
end


function calc_avg_n_spinfull(un, vn, nSites, E, T)
    avgNList = zeros(Float64, nSites)
    nStates = size(un, 1)
    @assert size(vn,1) == nStates
    @assert length(E) == nStates

    for i in 1:nSites
        avgN = 0.0
        for n in 1:nStates
            fn = f(E[n]; T=T)
            avgN += abs2(un[n, i]) * fn +
                    abs2(vn[n, i]) * (1 - fn)
        end
        avgNList[i] = avgN
    end

    return avgNList
end

calc_avg_n_spinfull (generic function with 1 method)

In [5]:
N             = 2
nSites        = 4 * N ^ 2

nUp          = ones(Float64, nSites)
nDn          = ones(Float64, nSites)

df = CSV.read("data/N$(N)/raw_df_square-octagon$N.csv", DataFrame)

x, y, neighbors = extract_geometry(df)
lambda = 0.0
lambdaIso = 0.0
t2 = 1.0

tMap = generate_t_map(neighbors, t2=t2)
twist = 0.0

isoMap = nothing

if lambda == 0.0 && twist == 0.0
    deltaOld      = ones(Float64, nSites)
    isComplexCalc = false
else
    deltaOld      = ones(ComplexF64, nSites)
    isComplexCalc = true
end

impuritySite  = 3
mu             = -0.2
U             = 3

J             = 0.0
K             = 0.0
T             = 0.0

tMatFileName = "data/N$(N)/ham_$(N)_t2_$(t2)"
tMat = generate_t_matrix(tMatFileName); ## One doesn't need to use prebuilt hamiltonian. dataframe is enough to construct this matrix

In [6]:
H1 = create_bdg_ham_up(deltaOld, copy(tMat), mu, nSites, nUp, nDn, U, impuritySite, J;
                       isComplex=isComplexCalc, includeHartree=false)

H2 = create_bdg_ham_dn(deltaOld, copy(tMat), mu, nSites, nUp, nDn, U, impuritySite, J;
                       isComplex=isComplexCalc, includeHartree=false);

(evals1, evecs1) = np.linalg.eigh(H1)
un_up, vn_dn = sort_evecs(evecs1, nSites);

(evals2, evecs2) = np.linalg.eigh(H2)
un_dn, vn_up = sort_evecs(evecs2, nSites);

evalPositive1 = evals1[nSites + 1:end]
evalPositive2 = evals2[nSites + 1:end]

deltaNew = calc_delta(U, un_up, vn_up, un_dn, vn_dn, nSites, evalPositive1, evalPositive2, T)
@show mean(abs.(deltaNew))

mean(abs.(deltaNew)) = 0.9304086344996013


0.9304086344996013

In [7]:
Hfull = create_bdg_ham_full(deltaOld, tMat, mu, nSites, nUp, nDn, U, impuritySite, J,
    x, y, neighbors, lambda, tMap)

(evals, evecs) = np.linalg.eigh(Hfull)

un_up, vn_dn, un_dn, vn_up, Epos = sort_evecs_spinfull(evecs, evals, nSites; tol=0.0)

deltaNew = calc_delta_spinfull(U, un_up, vn_up, un_dn, vn_dn, nSites, Epos, T)
@show mean(abs.(deltaNew))

mean(abs.(deltaNew)) = 0.9304086344996011


0.9304086344996011

In [8]:
"""
    run_self_consistency_numpy(deltaOld, mu, nSites, nUp, nDn, tMat, U, J, impuritySite, T;
                               tol=1e-5, maxCount=150, isComplexCalc=false, verboseLogIn=nothing)

Self-consistent non-spinful BdG using two decoupled 2nx2n sectors built from
`create_bdg_ham_up` and `create_bdg_ham_dn` and solved via NumPy `eigh`.

Returns:
(deltaFinal, nUpFinal, nDnFinal, nAvg, evecs_up, evecs_dn, evals_up, evals_dn,
 isConverged, endTime, count)
"""
function run_self_consistency_numpy(
    deltaOld, mu, nSites, nUp, nDn, tMat, U, J, impuritySite, T;
    tol=1e-5, maxCount=150, includeHartree=false, verboseLogIn=nothing
)
    startTime = time()

    deltaOld = copy(deltaOld)
    nUp = copy(nUp)
    nDn = copy(nDn)

    for count in 1:maxCount
        
        H1 = create_bdg_ham_up(
            deltaOld, copy(tMat), mu, nSites, nUp, nDn, U, impuritySite, J;
            isComplex=isComplexCalc
        )
        
        H2 = create_bdg_ham_dn(
            deltaOld, copy(tMat), mu, nSites, nUp, nDn, U, impuritySite, J;
            isComplex=isComplexCalc
        )

        evals1, evecs1 = np.linalg.eigh(H1)
        evals2, evecs2 = np.linalg.eigh(H2)

        un_up, vn_dn = sort_evecs(evecs1, nSites)
        un_dn, vn_up = sort_evecs(evecs2, nSites)

        evalPositive1 = evals1[nSites + 1:end]
        evalPositive2 = evals2[nSites + 1:end]

        deltaNew = calc_delta(
            U, un_up, vn_up, un_dn, vn_dn,
            nSites, evalPositive1, evalPositive2, T
        )

        nUpNew = calc_avg_n(un_up, vn_dn, nSites, evalPositive1, evalPositive2, T)
        nDnNew = calc_avg_n(un_dn, vn_up, nSites, evalPositive1, evalPositive2, T)

        if check_rel_tol(deltaOld, deltaNew; tol=tol)
            deltaFinal = deltaNew
            nUpFinal = nUpNew
            nDnFinal = nDnNew
            nAvg = nUpFinal + nDnFinal
            endTime = round(time() - startTime, digits=2)
            return deltaFinal, nUpFinal, nDnFinal, nAvg,
                   evecs1, evecs2, evals1, evals2,
                   true, endTime, count
        end

        deltaOld = deltaNew
        nUp = nUpNew
        nDn = nDnNew
    end

    deltaFinal = deltaOld
    nAvg = nUp + nDn
    endTime = round(time() - startTime, digits=2)
    return deltaFinal, nUp, nDn, nAvg,
           evecs1, evecs2, evals1, evals2,
           false, endTime, maxCount
end

run_self_consistency_numpy

In [9]:
function run_self_consistency_numpy_spinfull(
    deltaOld, mu, nSites, nUp, nDn, tMat, U, J,
    x, y, neighbors, lambda, lambdaIso, tMap, isoMap,
    impuritySite, T;
    tol=0.001, maxCount=100, K=0.0, includeHartree=false
)
    startTime = time()
    count = 0

    while true
        count += 1

        H_BdG = create_bdg_ham_full(
            deltaOld, tMat, mu, nSites, nUp, nDn, U, impuritySite, J,
            x, y, neighbors, lambda, tMap;
            K=K, includeHartree=includeHartree, isoMap=isoMap, lambdaIso=lambdaIso
        )

        (evals, evecs) = np.linalg.eigh(H_BdG)

        # positive-energy selection
        un_up, vn_dn, un_dn, vn_up, Epos = sort_evecs_spinfull(evecs, evals, nSites; tol=0.0)

        # gap update
        deltaNew = calc_delta_spinfull(U, un_up, vn_up, un_dn, vn_dn, nSites, Epos, T)

        # densities
        nUpNew = calc_avg_n_spinfull(un_up, vn_up, nSites, Epos, T)
        nDnNew = calc_avg_n_spinfull(un_dn, vn_dn, nSites, Epos, T)

        if check_rel_tol(deltaOld, deltaNew; tol=tol)
            deltaFinal = deltaNew
            nUpFinal = nUpNew
            nDnFinal = nDnNew
            nAvg = nUpFinal + nDnFinal
            isConverged = true
            endTime = round(time() - startTime, digits=2)

            return deltaFinal, nUpFinal, nDnFinal, nAvg,
                   evecs, evals, isConverged, endTime, count
        end

        deltaOld = deltaNew
        nUp = nUpNew
        nDn = nDnNew

        if count >= maxCount
            deltaFinal = deltaNew
            nUpFinal = nUpNew
            nDnFinal = nDnNew
            nAvg = nUpFinal + nDnFinal
            isConverged = false
            endTime = round(time() - startTime, digits=2)

            return deltaFinal, nUpFinal, nDnFinal, nAvg,
                   evecs, evals, isConverged, endTime, count
        end
    end
end

run_self_consistency_numpy_spinfull (generic function with 1 method)

In [10]:
deltaFinal, _, _, _, _, _, _, _, isConverged, endTime, count =
    run_self_consistency_numpy(
        deltaOld,
        mu,
        nSites,
        nUp,
        nDn,
        tMat,
        U,
        J,
        impuritySite,
        T
    )
@show mean(abs.(deltaFinal))

mean(abs.(deltaFinal)) = 0.8881666630824617


0.8881666630824617

In [11]:
deltaFinal, _, _, nAvg, evecs, evals, isConverged, endTime, count = 
    run_self_consistency_numpy_spinfull(
        deltaOld,
        mu, nSites,
        nUp, nDn,
        tMat, U, J,
        x, y, neighbors,
        lambda, lambdaIso,
        tMap, isoMap,
        impuritySite, T
    )

([0.8885633455353867, 0.8885633455353867, 0.8885633455353869, 0.8885633455353864, 0.888563345535387, 0.8885633455353853, 0.8885633455353866, 0.8885633455353861, 0.8885633455353864, 0.888563345535387, 0.8885633455353871, 0.8885633455353873, 0.8885633455353871, 0.8885633455353871, 0.8885633455353875, 0.8885633455353872], [0.5331773050915545, 0.5331773050915547, 0.5331773050915546, 0.5331773050915549, 0.5331773050915545, 0.5331773050915543, 0.5331773050915543, 0.5331773050915546, 0.5331773050915548, 0.5331773050915544, 0.5331773050915546, 0.5331773050915539, 0.5331773050915544, 0.5331773050915539, 0.5331773050915545, 0.533177305091554], [0.533177305091554, 0.5331773050915544, 0.5331773050915543, 0.5331773050915543, 0.5331773050915544, 0.5331773050915535, 0.5331773050915543, 0.533177305091554, 0.5331773050915539, 0.5331773050915537, 0.5331773050915539, 0.5331773050915544, 0.5331773050915537, 0.5331773050915544, 0.5331773050915539, 0.5331773050915539], [1.0663546101831085, 1.066354610183109

In [12]:
mean(abs.(deltaFinal))

0.8885633455353866

In [13]:
"""
    run_self_consistency_numpy_spinfull_twisted(
        deltaOld, mu, nSites, nUp, nDn, U, J, x, y,
        neighbors, lambda, lambdaIso, tMap, isoMap, impuritySite, T;
        twist=0.0, K=0.0, tol=1e-3, maxCount=200,
        includeHartree=false, verboseLogIn=nothing
    )

Self-consistency loop for the spinful BdG Hamiltonian with optional twist and SOC.

Uses the 4n x 4n BdG Hamiltonian in the basis
(c_up; c_dn^dagger; c_dn; c_up^dagger), selects the positive-energy
states using `evals .> 0`, and updates the gap and spin densities.

Returns:
`deltaFinal, nUpFinal, nDnFinal, nAvg, evecs, evals, isConverged, endTime, count`
"""
function run_self_consistency_numpy_spinfull_twisted(
    deltaOld, mu, nSites, nUp, nDn, U, J, x, y,
    neighbors, lambda, lambdaIso, tMap, isoMap, impuritySite, T;
    twist=0.0, K=0.0, tol=1e-3, maxCount=200,
    includeHartree::Bool=false, verboseLogIn=nothing
)
    startTime = time()
    count = 0

    while true
        count += 1
        innerLoopTime = time()

        H_BdG = create_bdg_ham_full_twisted(
            deltaOld, mu, nSites,
            nUp, nDn, U, impuritySite, J,
            x, y, neighbors, lambda, tMap;
            twist=twist, K=K, includeHartree=includeHartree,
            isoMap=isoMap, lambdaIso=lambdaIso
        )

        (evals, evecs) = np.linalg.eigh(H_BdG)

        # positive-energy selection + block slicing
        un_up, vn_dn, un_dn, vn_up, Epos = sort_evecs_spinfull(evecs, evals, nSites; tol=0.0)

        # gap update (uses v_up with u_dn etc. as in your definition)
        deltaNew = calc_delta_spinfull(U, un_up, vn_up, un_dn, vn_dn, nSites, Epos, T)

        # densities (IMPORTANT: n_up uses v_up, n_dn uses v_dn)
        nUpNew = calc_avg_n_spinfull(un_up, vn_up, nSites, Epos, T)
        nDnNew = calc_avg_n_spinfull(un_dn, vn_dn, nSites, Epos, T)

        if check_rel_tol(deltaOld, deltaNew; tol=tol)
            deltaFinal = deltaNew
            nUpFinal = nUpNew
            nDnFinal = nDnNew
            nAvg = nUpFinal + nDnFinal
            isConverged = true
            endTime = round(time() - startTime, digits=2)

            return deltaFinal, nUpFinal, nDnFinal, nAvg,
                   evecs, evals, isConverged, endTime, count
        end

        # update for next iteration
        deltaOld = deltaNew
        nUp = nUpNew
        nDn = nDnNew

        if verboseLogIn !== nothing
            elapsed = round(time() - innerLoopTime, digits=2)
            message = string("Iteration $count completed in ", format_elapsed_time(elapsed))
            write_log(verboseLogIn, message)
        end

        if count >= maxCount
            deltaFinal = deltaNew
            nUpFinal = nUpNew
            nDnFinal = nDnNew
            nAvg = nUpFinal + nDnFinal
            isConverged = false
            endTime = round(time() - startTime, digits=2)

            return deltaFinal, nUpFinal, nDnFinal, nAvg,
                   evecs, evals, isConverged, endTime, count
        end
    end
end

run_self_consistency_numpy_spinfull_twisted

In [14]:
deltaFinal, _, _, nAvg, evecs, evals, isConverged, endTime, count = 
    run_self_consistency_numpy_spinfull_twisted(
        deltaOld,
        mu, nSites,
        nUp, nDn,
        U, J, x, y,
        neighbors,
        lambda, lambdaIso,
        tMap, isoMap,
        impuritySite, T,
        twist=twist
    )

([0.8885633455353867, 0.8885633455353867, 0.8885633455353869, 0.8885633455353864, 0.888563345535387, 0.8885633455353853, 0.8885633455353866, 0.8885633455353861, 0.8885633455353864, 0.888563345535387, 0.8885633455353871, 0.8885633455353873, 0.8885633455353871, 0.8885633455353871, 0.8885633455353875, 0.8885633455353872], [0.5331773050915545, 0.5331773050915547, 0.5331773050915546, 0.5331773050915549, 0.5331773050915545, 0.5331773050915543, 0.5331773050915543, 0.5331773050915546, 0.5331773050915548, 0.5331773050915544, 0.5331773050915546, 0.5331773050915539, 0.5331773050915544, 0.5331773050915539, 0.5331773050915545, 0.533177305091554], [0.533177305091554, 0.5331773050915544, 0.5331773050915543, 0.5331773050915543, 0.5331773050915544, 0.5331773050915535, 0.5331773050915543, 0.533177305091554, 0.5331773050915539, 0.5331773050915537, 0.5331773050915539, 0.5331773050915544, 0.5331773050915537, 0.5331773050915544, 0.5331773050915539, 0.5331773050915539], [1.0663546101831085, 1.066354610183109

In [15]:
mean(abs.(deltaFinal))

0.8885633455353866